# S4 Portfolio — Integrated Impact Assessment

**Interactive tier-by-tier supply-chain analysis with scenario and dependency weighting.**

Passes:
- **Tier 0** — Direct investment: CAPEX split across 8 supplying sectors
- **Tier 1** — First upstream: bilateral sourcing-country breakdown
- **Tier 2** — Second upstream: sub-suppliers of Tier 1
- **Tier 3–10** — Deep upstream: aggregated residual signal
- **Scenario adjustment** — OSeMOSYS SSP1–5 intensity trajectories
- **Dependency weighting** — ENCORE materiality × WWF risk × SC sector sensitivity

**Positive impacts** (green): Employment, Value Added, Avoided CO₂, Beneficiaries, Rail reach  
**Negative impacts** (red/orange): GHG emissions, Water withdrawal, NOx

---

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PARAMETERS — edit these cells to change the analysis
# ═══════════════════════════════════════════════════════════════════════
from pathlib import Path

# ── Input files ──────────────────────────────────────────────────────
ROOT        = Path(".").resolve().parent          # repo root
INPUT_DIR   = Path("modeled_input_data")           # relative to notebook
RESULTS_DIR = Path("results")                      # pre-computed by assess.py

HOSPITALS_FILE = INPUT_DIR / "hospitals_finance_input.csv"
HYDRO_FILE     = INPUT_DIR / "hydro_finance_input.csv"
RAIL_FILE      = INPUT_DIR / "rail_finance_input.csv"

# ── Scenario parameters ───────────────────────────────────────────────
SCENARIO     = "SSP2-4.5"   # baseline scenario for single-scenario views
COMPARE_SSPS = ["SSP1-1.9", "SSP2-4.5", "SSP3-7.0", "SSP5-8.5"]  # for comparison
YEARS        = [2025, 2030, 2040]
FOCUS_YEAR   = 2030

# ── Display parameters ────────────────────────────────────────────────
TOP_N_SECTORS = 6   # how many supplying sectors to show in bar charts
FIG_WIDTH     = 14

print(f"Parameters loaded.  Scenario: {SCENARIO}  |  Focus year: {FOCUS_YEAR}")
print(f"Input dir: {INPUT_DIR.resolve()}")
print(f"Results dir: {RESULTS_DIR.resolve()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# IMPORTS & STYLE
# ═══════════════════════════════════════════════════════════════════════
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.3,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})

# ── Colour palette ────────────────────────────────────────────────────
NEG_COL   = "#d62728"   # red  — negative impacts (GHG, water, NOx)
POS_COL   = "#2ca02c"   # green — positive impacts (jobs, VA, avoided CO2)
NEU_COL   = "#1f77b4"   # blue  — neutral (spend, investment)
WAT_COL   = "#17becf"   # cyan  — water
EMP_COL   = "#9467bd"   # purple — employment

# Region colours
REGION_COLS = {
    "Africa": "#e6550d", "Asia": "#fd8d3c",
    "Europe": "#3182bd", "LATAM": "#31a354",
    "Global": "#969696",
}

# Supplying sector colours (8 tvp_io_lib sectors)
SECTOR_COLS = {
    "Construction":        "#8c564b",
    "Energy_Utilities":    "#ff7f0e",
    "Manufacturing":       "#1f77b4",
    "Transport_Logistics": "#2ca02c",
    "Health_Social":       "#9467bd",
    "Agriculture":         "#bcbd22",
    "Mining_Extraction":   "#7f7f7f",
    "Water_Waste":         "#17becf",
}

# Impact polarity
IMPACT_SIGN = {
    "GHG_tCO2e":           ("negative", "GHG (tCO2e)",        NEG_COL),
    "Water_1000m3":        ("negative", "Water (000 m³)",     WAT_COL),
    "NOx_t":               ("negative", "NOx (t)",            "#d6616b"),
    "Employment_FTE":      ("positive", "Jobs (FTE)",         EMP_COL),
    "ValueAdded_M$":       ("positive", "Value Added (M USD)",POS_COL),
    "avoided_CO2_tCO2e":   ("positive", "Avoided CO₂ (tCO2e)","#00a651"),
    "beneficiaries":       ("positive", "Beneficiaries",      "#4dac26"),
    "reach_ppl_yr":        ("positive", "Rail reach (ppl/yr)","#6baed6"),
}

def fmt_k(v):
    """Format large numbers with k/M suffix."""
    if abs(v) >= 1e6: return f"{v/1e6:.1f}M"
    if abs(v) >= 1e3: return f"{v/1e3:.0f}k"
    return f"{v:.1f}"

print("Imports and style loaded.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# LOAD DATA — input files + pre-computed results
# ═══════════════════════════════════════════════════════════════════════

# ── Input files ──────────────────────────────────────────────────────
hospitals = pd.read_csv(HOSPITALS_FILE)
hydro     = pd.read_csv(HYDRO_FILE)
rail      = pd.read_csv(RAIL_FILE)

# ── Positive outcomes (generated by assess.py) ────────────────────────
pos_out = pd.read_csv(RESULTS_DIR / "positive_outcomes.csv")

# ── Supply-chain tier CSVs ────────────────────────────────────────────
t0  = pd.read_csv(RESULTS_DIR / "supply_chain_tier0.csv")
t1  = pd.read_csv(RESULTS_DIR / "supply_chain_tier1.csv")
t2  = pd.read_csv(RESULTS_DIR / "supply_chain_tier2.csv")
t3  = pd.read_csv(RESULTS_DIR / "supply_chain_tier3_10.csv")
sc_sum = pd.read_csv(RESULTS_DIR / "supply_chain_summary.csv")

# ── Scenario-weighted and dependency-weighted ─────────────────────────
sw  = pd.read_csv(RESULTS_DIR / "scenario_weighted_summary.csv")
dw  = pd.read_csv(RESULTS_DIR / "dep_weighted_summary.csv")
dwt = pd.read_csv(RESULTS_DIR / "dep_weighted_tiers.csv")

# Normalise region capitalisation
for df in [t0, t1, t2, t3, sc_sum, sw, dw, dwt]:
    if "region" in df.columns:
        df["region"] = df["region"].str.strip().str.title()
        df["region"] = df["region"].replace({"Latam": "LATAM"})

# Project metadata lookup
PROJ_META = sc_sum.set_index("project_id")[["region", "asset_class", "sector_code"]].to_dict("index")

print(f"Input files loaded: {len(hospitals)} hospitals, {len(hydro)} hydro, {len(rail)} rail")
print(f"Supply-chain rows  — T0:{len(t0)}  T1:{len(t1)}  T2:{len(t2)}  T3-10:{len(t3)}")
print(f"Dep-weighted rows  — {len(dwt)} total")

---
## 1. Portfolio Overview — Positive vs Negative Impacts

In [ ]:
# ── 1.1 Net impact ledger ─────────────────────────────────────────────
ghg_col = [c for c in sc_sum.columns if c.startswith("GHG_tCO2e")][0]
emp_col = [c for c in sc_sum.columns if c.startswith("Emp_FTE")][0]
wat_col = [c for c in sc_sum.columns if c.startswith("Water_1000m3")][0]
va_col  = [c for c in sc_sum.columns if c.startswith("VA_Musd")][0]

ledger = sc_sum[["project_id", "region", "asset_class", ghg_col, emp_col, wat_col, va_col]].copy()
ledger.columns = ["Project", "Region", "Asset", "[−] GHG tCO2e", "[+] Jobs FTE",
                  "[−] Water 000m³", "[+] VA M USD"]
ledger = ledger.merge(pos_out[["project_id", "avoided_CO2_tCO2e", "beneficiaries", "reach_ppl_yr"]],
                      left_on="Project", right_on="project_id", how="left").drop(columns="project_id")
ledger.rename(columns={"avoided_CO2_tCO2e": "[+] Avoided CO₂",
                        "beneficiaries":     "[+] Beneficiaries",
                        "reach_ppl_yr":      "[+] Rail reach"}, inplace=True)
ledger["[+] Avoided CO₂"]    = ledger["[+] Avoided CO₂"].fillna(0)
ledger["[+] Beneficiaries"]  = ledger["[+] Beneficiaries"].fillna(0).astype(int)
ledger["[+] Rail reach"]     = ledger["[+] Rail reach"].fillna(0).astype(int)

# Net GHG = supply-chain emissions - avoided
ledger["Net GHG tCO2e"] = ledger["[−] GHG tCO2e"] - ledger["[+] Avoided CO₂"]

pd.set_option("display.float_format", "{:,.0f}".format)
pd.set_option("display.max_columns", 20)
ledger.sort_values("[−] GHG tCO2e", ascending=False)

In [ ]:
# ── 1.2 Visual: positive vs negative by project ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, 5))
projects  = sc_sum.sort_values(ghg_col, ascending=False)["project_id"].tolist()

# Left: negative impacts (GHG + water)
ax = axes[0]
ghg_vals = sc_sum.set_index("project_id").loc[projects, ghg_col] / 1e3
wat_vals = sc_sum.set_index("project_id").loc[projects, wat_col]

x = np.arange(len(projects))
b1 = ax.bar(x, ghg_vals, color=NEG_COL, alpha=0.85, label="GHG (ktCO2e)")
ax2_twin = ax.twinx()
ax2_twin.plot(x, wat_vals, "o--", color=WAT_COL, linewidth=1.5, markersize=5, label="Water (000 m³)")
ax2_twin.set_ylabel("Water withdrawal (000 m³)", color=WAT_COL)
ax2_twin.tick_params(axis="y", colors=WAT_COL)
ax2_twin.spines[["top", "right"]].set_visible(False)
ax.set_xticks(x); ax.set_xticklabels(projects, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("GHG (ktCO2e)"); ax.set_title("[−] Negative Impacts", color=NEG_COL, fontweight="bold")
lines1, _ = ax.get_legend_handles_labels()
lines2, _ = ax2_twin.get_legend_handles_labels()
ax.legend(lines1 + lines2, ["GHG (ktCO2e)", "Water (000 m³)"], fontsize=8, loc="upper right")

# Right: positive impacts (jobs + avoided CO2)
ax = axes[1]
emp_vals = sc_sum.set_index("project_id").loc[projects, emp_col]
avo_vals = pos_out.set_index("project_id").reindex(projects)["avoided_CO2_tCO2e"].fillna(0) / 1e3

ax.bar(x - 0.2, emp_vals, width=0.4, color=EMP_COL, alpha=0.85, label="Jobs (FTE)")
ax3_twin = ax.twinx()
ax3_twin.bar(x + 0.2, avo_vals, width=0.4, color=POS_COL, alpha=0.75, label="Avoided CO₂ (ktCO2e)")
ax3_twin.set_ylabel("Avoided CO₂ (ktCO2e)", color=POS_COL)
ax3_twin.tick_params(axis="y", colors=POS_COL)
ax3_twin.spines[["top", "right"]].set_visible(False)
ax.set_xticks(x); ax.set_xticklabels(projects, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("Employment (FTE)", color=EMP_COL)
ax.tick_params(axis="y", colors=EMP_COL)
ax.set_title("[+] Positive Impacts", color=POS_COL, fontweight="bold")
p1 = mpatches.Patch(color=EMP_COL, label="Jobs (FTE)")
p2 = mpatches.Patch(color=POS_COL, label="Avoided CO₂ (ktCO2e)")
ax.legend(handles=[p1, p2], fontsize=8, loc="upper right")

fig.suptitle("Portfolio — Negative vs Positive Supply-Chain Impacts (Tiers 0–10)",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 2. Tier 0 — Direct Investment (CAPEX)

In [ ]:
# ── 2.1 Tier 0: per-sector GHG and jobs, by region ────────────────────
fig, axes = plt.subplots(2, 2, figsize=(FIG_WIDTH, 9))
axes = axes.flatten()

indicators = [
    ("GHG_tCO2e",    "GHG (tCO2e)",    NEG_COL,  "[−]"),
    ("Water_1000m3", "Water (000 m³)", WAT_COL,  "[−]"),
    ("Employment_FTE","Jobs (FTE)",    EMP_COL,  "[+]"),
    ("ValueAdded_M$", "VA (M USD)",    POS_COL,  "[+]"),
]

regions  = ["Africa", "Asia", "Europe", "LATAM"]
sectors  = t0["supplying_sector"].unique()

for ax, (col, label, color, sign) in zip(axes, indicators):
    # Pivot: rows = supplying_sector, columns = region
    piv = (t0.groupby(["supplying_sector", "region"])[col]
              .sum().unstack(fill_value=0))
    # Keep only regions present
    reg_cols = [r for r in regions if r in piv.columns]
    piv = piv[reg_cols].head(TOP_N_SECTORS)

    x = np.arange(len(piv))
    w = 0.8 / len(reg_cols)
    for i, reg in enumerate(reg_cols):
        ax.bar(x + i * w, piv[reg], width=w,
               color=REGION_COLS.get(reg, "#999"), alpha=0.85, label=reg)

    ax.set_xticks(x + w * (len(reg_cols) - 1) / 2)
    ax.set_xticklabels(piv.index, rotation=30, ha="right", fontsize=8)
    ax.set_title(f"Tier 0  {sign}  {label}",
                 color=color, fontweight="bold")
    ax.set_ylabel(label)
    ax.legend(fontsize=7, loc="upper right")

fig.suptitle("Tier 0 — Direct CAPEX Impact by Supplying Sector & Region",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.2 Tier 0: stacked bars by project showing sector breakdown ───────
fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, 5))

for ax, (col, label, color, sign) in zip(axes, [
    ("GHG_tCO2e", "GHG (tCO2e)", NEG_COL, "[−]"),
    ("Employment_FTE", "Jobs (FTE)", EMP_COL, "[+]"),
]):
    piv = (t0.groupby(["project_id", "supplying_sector"])[col]
              .sum().unstack(fill_value=0))
    piv = piv.sort_values(piv.columns.tolist(), ascending=False)
    bottom = np.zeros(len(piv))
    for sec in piv.columns:
        vals = piv[sec].values
        ax.bar(piv.index, vals, bottom=bottom,
               color=SECTOR_COLS.get(sec, "#ccc"), alpha=0.88, label=sec, width=0.6)
        bottom += vals

    # Annotate region above each bar
    for pid, total in zip(piv.index, bottom):
        reg = PROJ_META.get(pid, {}).get("region", "")
        ax.text(pid, total * 1.01, reg, ha="center", va="bottom", fontsize=6.5, color="#555")

    ax.set_title(f"Tier 0  {sign}  {label} by Sector", color=color, fontweight="bold")
    ax.set_ylabel(label)
    ax.set_xticklabels(piv.index, rotation=35, ha="right", fontsize=8)
    ax.legend(fontsize=6.5, loc="upper right", ncol=2)

plt.tight_layout()
plt.show()

---
## 3. Tier 1 — First Upstream (Bilateral Sourcing)

In [ ]:
# ── 3.1 Tier 1: sourcing-region map — where do supply inputs come from?
fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, 5))

for ax, (col, label, color, sign) in zip(axes, [
    ("GHG_tCO2e",    "GHG (tCO2e)",  NEG_COL, "[−]"),
    ("Employment_FTE","Jobs (FTE)",   EMP_COL, "[+]"),
]):
    piv = (t1.groupby(["project_id", "sourcing_region"])[col]
              .sum().unstack(fill_value=0))
    piv = piv[[c for c in ["Africa","Asia","Europe","LATAM","Global"] if c in piv.columns]]

    bottom = np.zeros(len(piv))
    for reg in piv.columns:
        vals = piv[reg].values
        ax.bar(piv.index, vals, bottom=bottom,
               color=REGION_COLS.get(reg, "#ccc"), alpha=0.88, label=reg, width=0.6)
        bottom += vals

    ax.set_title(f"Tier 1  {sign}  {label}\nby Sourcing Region (bilateral)",
                 color=color, fontweight="bold")
    ax.set_ylabel(label)
    ax.set_xticklabels(piv.index, rotation=35, ha="right", fontsize=8)
    ax.legend(title="Sourcing region", fontsize=7, loc="upper right")

fig.suptitle("Tier 1 — First Upstream: Bilateral Sourcing-Country Impact",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.2 Tier 1: GHG by supplying sector × sourcing region (heatmap) ───
piv_heat = (t1.groupby(["supplying_sector", "sourcing_region"])["GHG_tCO2e"]
               .sum().unstack(fill_value=0))
piv_heat = piv_heat[[c for c in ["Africa","Asia","Europe","LATAM","Global"] if c in piv_heat.columns]]

fig, ax = plt.subplots(figsize=(9, 4))
cmap = LinearSegmentedColormap.from_list("wr", ["#fff5f0", NEG_COL])
im = ax.imshow(piv_heat.values, cmap=cmap, aspect="auto")
ax.set_xticks(range(len(piv_heat.columns))); ax.set_xticklabels(piv_heat.columns, fontsize=9)
ax.set_yticks(range(len(piv_heat.index)));  ax.set_yticklabels(piv_heat.index, fontsize=9)

for i in range(len(piv_heat.index)):
    for j in range(len(piv_heat.columns)):
        val = piv_heat.iloc[i, j]
        ax.text(j, i, fmt_k(val), ha="center", va="center",
                fontsize=7.5, color="white" if val > piv_heat.values.max() * 0.5 else "black")

plt.colorbar(im, ax=ax, label="GHG tCO2e", shrink=0.7)
ax.set_title("Tier 1 [−] GHG — Supplying Sector × Sourcing Region",
             color=NEG_COL, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3 Tier 1: Water heatmap (sourcing sector × region) ──────────────
piv_w = (t1.groupby(["supplying_sector", "sourcing_region"])["Water_1000m3"]
            .sum().unstack(fill_value=0))
piv_w = piv_w[[c for c in ["Africa","Asia","Europe","LATAM","Global"] if c in piv_w.columns]]

fig, ax = plt.subplots(figsize=(9, 4))
cmap_w = LinearSegmentedColormap.from_list("wc", ["#f0f9ff", WAT_COL])
im = ax.imshow(piv_w.values, cmap=cmap_w, aspect="auto")
ax.set_xticks(range(len(piv_w.columns))); ax.set_xticklabels(piv_w.columns, fontsize=9)
ax.set_yticks(range(len(piv_w.index)));  ax.set_yticklabels(piv_w.index, fontsize=9)

for i in range(len(piv_w.index)):
    for j in range(len(piv_w.columns)):
        val = piv_w.iloc[i, j]
        ax.text(j, i, fmt_k(val), ha="center", va="center",
                fontsize=7.5, color="white" if val > piv_w.values.max() * 0.5 else "black")

plt.colorbar(im, ax=ax, label="Water (000 m³)", shrink=0.7)
ax.set_title("Tier 1 [−] Water Withdrawal — Supplying Sector × Sourcing Region",
             color=WAT_COL, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. Tier 2 — Second Upstream

In [ ]:
# ── 4.1 Tier 2: sector × region bar charts ────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(FIG_WIDTH, 9))
axes = axes.flatten()

indicators_t2 = [
    ("GHG_tCO2e",    "GHG (tCO2e)",   NEG_COL, "[−]"),
    ("Water_1000m3", "Water (000 m³)",WAT_COL,  "[−]"),
    ("Employment_FTE","Jobs (FTE)",   EMP_COL,  "[+]"),
    ("ValueAdded_M$", "VA (M USD)",   POS_COL,  "[+]"),
]

for ax, (col, label, color, sign) in zip(axes, indicators_t2):
    piv = (t2.groupby(["supplying_sector", "region"])[col]
              .sum().unstack(fill_value=0))
    reg_cols = [r for r in ["Africa","Asia","Europe","LATAM"] if r in piv.columns]
    piv = piv[reg_cols].nlargest(TOP_N_SECTORS, reg_cols[0] if reg_cols else piv.columns[0])

    x = np.arange(len(piv))
    w = 0.8 / len(reg_cols)
    for i, reg in enumerate(reg_cols):
        ax.bar(x + i * w, piv[reg], width=w,
               color=REGION_COLS.get(reg, "#999"), alpha=0.85, label=reg)
    ax.set_xticks(x + w * (len(reg_cols) - 1) / 2)
    ax.set_xticklabels(piv.index, rotation=30, ha="right", fontsize=8)
    ax.set_title(f"Tier 2  {sign}  {label}", color=color, fontweight="bold")
    ax.set_ylabel(label)
    ax.legend(fontsize=7, loc="upper right")

fig.suptitle("Tier 2 — Second Upstream by Sector & Region",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Tier 3–10 — Deep Upstream

In [ ]:
# ── 5.1 Tier 3-10: sector × region ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(FIG_WIDTH, 9))
axes = axes.flatten()

for ax, (col, label, color, sign) in zip(axes, [
    ("GHG_tCO2e",    "GHG (tCO2e)",   NEG_COL, "[−]"),
    ("Water_1000m3", "Water (000 m³)",WAT_COL,  "[−]"),
    ("Employment_FTE","Jobs (FTE)",   EMP_COL,  "[+]"),
    ("ValueAdded_M$", "VA (M USD)",   POS_COL,  "[+]"),
]):
    piv = (t3.groupby(["supplying_sector", "region"])[col]
              .sum().unstack(fill_value=0))
    reg_cols = [r for r in ["Africa","Asia","Europe","LATAM"] if r in piv.columns]
    piv = piv[reg_cols]

    x = np.arange(len(piv))
    w = 0.8 / max(len(reg_cols), 1)
    for i, reg in enumerate(reg_cols):
        ax.bar(x + i * w, piv[reg], width=w,
               color=REGION_COLS.get(reg, "#999"), alpha=0.85, label=reg)
    ax.set_xticks(x + w * (len(reg_cols) - 1) / 2)
    ax.set_xticklabels(piv.index, rotation=30, ha="right", fontsize=8)
    ax.set_title(f"Tier 3–10  {sign}  {label}", color=color, fontweight="bold")
    ax.set_ylabel(label)
    ax.legend(fontsize=7, loc="upper right")

fig.suptitle("Tiers 3–10 — Deep Upstream (Aggregated) by Sector & Region",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Cross-Tier Comparison — Cumulative Impact by Tier

In [ ]:
# ── 6.1 Stacked waterfall: tier contribution per indicator ─────────────
# Label each tier DF and stack
t0c = t0.copy(); t0c["tier_label"] = "Tier 0"
t1c = t1.copy(); t1c["tier_label"] = "Tier 1"
t2c = t2.copy(); t2c["tier_label"] = "Tier 2"
t3c = t3.copy(); t3c["tier_label"] = "Tier 3-10"
all_tiers = pd.concat([t0c, t1c, t2c, t3c], ignore_index=True)

tier_order = ["Tier 0", "Tier 1", "Tier 2", "Tier 3-10"]
tier_cols  = ["#c6dbef", "#6baed6", "#2171b5", "#08306b"]

fig, axes = plt.subplots(1, 4, figsize=(FIG_WIDTH, 5))

for ax, (col, label, color, sign) in zip(axes, [
    ("GHG_tCO2e",    "GHG (tCO2e)",   NEG_COL,  "[−]"),
    ("Water_1000m3", "Water (000 m³)",WAT_COL,   "[−]"),
    ("Employment_FTE","Jobs (FTE)",   EMP_COL,   "[+]"),
    ("ValueAdded_M$", "VA (M USD)",   POS_COL,   "[+]"),
]):
    tier_sums = all_tiers.groupby("tier_label")[col].sum().reindex(tier_order)
    bars = ax.bar(tier_sums.index, tier_sums.values, color=tier_cols, alpha=0.9, width=0.6)
    for bar, val in zip(bars, tier_sums.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.01, fmt_k(val),
                ha="center", va="bottom", fontsize=8)
    ax.set_title(f"{sign} {label}", color=color, fontweight="bold")
    ax.set_xticklabels(tier_sums.index, rotation=20, ha="right", fontsize=8)
    ax.set_ylabel(label)

fig.suptitle("Portfolio Total by Tier — All Projects", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.2 GHG waterfall: cumulative tier stack per project ──────────────
fig, ax = plt.subplots(figsize=(FIG_WIDTH, 5))

projects_ord = sc_sum.sort_values(ghg_col, ascending=False)["project_id"].tolist()
x = np.arange(len(projects_ord))

tier_labels = ["Tier 0", "Tier 1", "Tier 2", "Tier 3-10"]
bottoms = np.zeros(len(projects_ord))

for tlab, tdf, tc in zip(tier_labels, [t0c, t1c, t2c, t3c], tier_cols):
    vals = tdf.groupby("project_id")["GHG_tCO2e"].sum().reindex(projects_ord, fill_value=0).values
    ax.bar(x, vals, bottom=bottoms, color=tc, alpha=0.9, label=tlab, width=0.6)
    bottoms += vals

# Overlay avoided CO2 as negative bars
for i, pid in enumerate(projects_ord):
    po_row = pos_out[pos_out["project_id"] == pid]
    if not po_row.empty and po_row.iloc[0]["avoided_CO2_tCO2e"] > 0:
        avoided = po_row.iloc[0]["avoided_CO2_tCO2e"]
        ax.bar(i, -avoided, bottom=0, color=POS_COL, alpha=0.85, width=0.6,
               label="Avoided CO₂" if i == 0 else "")
        ax.text(i, -avoided * 1.05, f"-{fmt_k(avoided)}",
                ha="center", va="top", fontsize=7, color=POS_COL)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(projects_ord, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("GHG (tCO2e)")
ax.set_title("[−] Supply-chain GHG vs [+] Avoided CO₂ — Tier Stack per Project",
             fontweight="bold")

# Add region label
for i, pid in enumerate(projects_ord):
    reg = PROJ_META.get(pid, {}).get("region", "")
    ax.text(i, bottoms[i] * 1.01, reg, ha="center", va="bottom", fontsize=6.5, color="#444")

ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

---
## 7. Region × Sector Impact Heatmaps (All Tiers)

In [ ]:
# ── 7.1 Full portfolio: GHG heatmap region × supplying sector ─────────
all_tiers_full = all_tiers.copy()
# Normalise region
all_tiers_full["region"] = all_tiers_full["region"].str.strip().str.title()
all_tiers_full["region"] = all_tiers_full["region"].replace({"Latam": "LATAM"})

fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, 4))

for ax, (col, label, cmap_colors, sign) in zip(axes, [
    ("GHG_tCO2e",    "GHG (tCO2e)",   ["#fff5f0", NEG_COL],   "[−]"),
    ("Employment_FTE","Jobs (FTE)",   ["#f7fcf5", EMP_COL],   "[+]"),
]):
    piv = (all_tiers_full.groupby(["region", "supplying_sector"])[col]
                          .sum().unstack(fill_value=0))
    piv = piv[[c for c in [
        "Construction","Energy_Utilities","Manufacturing",
        "Transport_Logistics","Health_Social","Agriculture",
        "Mining_Extraction","Water_Waste"] if c in piv.columns]]

    cmap_h = LinearSegmentedColormap.from_list("hm", cmap_colors)
    im = ax.imshow(piv.values, cmap=cmap_h, aspect="auto")
    ax.set_xticks(range(len(piv.columns)))
    ax.set_xticklabels(piv.columns, rotation=35, ha="right", fontsize=8)
    ax.set_yticks(range(len(piv.index)))
    ax.set_yticklabels(piv.index, fontsize=9)

    for i in range(len(piv.index)):
        for j in range(len(piv.columns)):
            v = piv.iloc[i, j]
            ax.text(j, i, fmt_k(v), ha="center", va="center", fontsize=7,
                    color="white" if v > piv.values.max() * 0.55 else "black")

    plt.colorbar(im, ax=ax, label=label, shrink=0.85)
    ax.set_title(f"All Tiers  {sign}  {label}\nRegion × Supplying Sector",
                 color=cmap_colors[1], fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# ── 7.2 Per-tier: GHG region × sector (faceted) ───────────────────────
fig, axes = plt.subplots(1, 4, figsize=(FIG_WIDTH, 4.5))

for ax, (tdf, tlab) in zip(axes, [
    (t0c, "Tier 0"), (t1c, "Tier 1"), (t2c, "Tier 2"), (t3c, "Tier 3-10")
]):
    # Use sourcing_region for tier1, region for others
    reg_col = "sourcing_region" if (tlab == "Tier 1" and "sourcing_region" in tdf.columns) else "region"
    tdf_r = tdf.copy()
    tdf_r["_r"] = tdf_r[reg_col].str.strip().str.title().replace({"Latam": "LATAM"})
    piv = (tdf_r.groupby(["_r", "supplying_sector"])["GHG_tCO2e"]
                .sum().unstack(fill_value=0))
    piv = piv[[c for c in ["Construction","Energy_Utilities","Manufacturing",
                            "Transport_Logistics","Agriculture","Mining_Extraction",
                            "Health_Social","Water_Waste"] if c in piv.columns]]
    if piv.empty:
        ax.set_title(f"{tlab}\n(no data)"); continue

    cmap_h = LinearSegmentedColormap.from_list("hm", ["#fff5f0", NEG_COL])
    im = ax.imshow(piv.values, cmap=cmap_h, aspect="auto")
    ax.set_xticks(range(len(piv.columns)))
    ax.set_xticklabels(piv.columns, rotation=40, ha="right", fontsize=7)
    ax.set_yticks(range(len(piv.index)))
    ax.set_yticklabels(piv.index, fontsize=8)

    for i in range(len(piv.index)):
        for j in range(len(piv.columns)):
            v = piv.iloc[i, j]
            if v > 0:
                ax.text(j, i, fmt_k(v), ha="center", va="center", fontsize=6,
                        color="white" if v > piv.values.max() * 0.5 else "black")

    label_suffix = " (sourcing region)" if tlab == "Tier 1" else " (project region)"
    ax.set_title(f"{tlab}\n[−] GHG{label_suffix}", color=NEG_COL, fontweight="bold", fontsize=9)

plt.suptitle("GHG by Region × Sector — Per Tier", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Scenario Analysis — SSP Trajectories

In [ ]:
# ── 8.1 Portfolio GHG trajectory: baseline vs SSP scenarios ───────────
fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, 5))

ssp_cols = {
    "SSP1-1.9": "#2ca02c",
    "SSP2-4.5": "#1f77b4",
    "SSP3-7.0": "#ff7f0e",
    "SSP4-6.0": "#9467bd",
    "SSP5-8.5": "#d62728",
}

ax = axes[0]
for ssp, scolor in ssp_cols.items():
    sub = sw[sw["scenario"] == ssp].groupby("year")["GHG_adj_tCO2e"].sum().reset_index()
    if sub.empty: continue
    ax.plot(sub["year"], sub["GHG_adj_tCO2e"] / 1e3, "o-",
            color=scolor, linewidth=2, markersize=5, label=ssp)

# Baseline (2020)
baseline_ghg = sw.groupby("project_id")["GHG_tCO2e"].first().sum()
ax.axhline(baseline_ghg / 1e3, color="black", linestyle="--", linewidth=1.2, label="2020 baseline")
ax.set_xlabel("Year"); ax.set_ylabel("Portfolio GHG (ktCO2e)")
ax.set_title("[−] Portfolio GHG Trajectory\nScenario-adjusted (all projects)",
             color=NEG_COL, fontweight="bold")
ax.legend(fontsize=8)

# Right: Employment trajectory
ax = axes[1]
for ssp, scolor in ssp_cols.items():
    sub = sw[sw["scenario"] == ssp].groupby("year")["Employment_adj_FTE"].sum().reset_index()
    if sub.empty: continue
    ax.plot(sub["year"], sub["Employment_adj_FTE"] / 1e3, "o-",
            color=scolor, linewidth=2, markersize=5, label=ssp)

base_emp = sw.groupby("project_id")["Employment_FTE"].first().sum()
ax.axhline(base_emp / 1e3, color="black", linestyle="--", linewidth=1.2, label="2020 baseline")
ax.set_xlabel("Year"); ax.set_ylabel("Portfolio Employment (k FTE)")
ax.set_title("[+] Portfolio Jobs Trajectory\nScenario-adjusted (renewable-jobs premium)",
             color=EMP_COL, fontweight="bold")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── 8.2 SSP comparison at focus year — all stressors by project ────────
ssps_plot = [s for s in COMPARE_SSPS if s in sw["scenario"].unique()]
n_ssps = len(ssps_plot)

fig, axes = plt.subplots(2, 2, figsize=(FIG_WIDTH, 9))
axes = axes.flatten()

for ax, (col, label, color, sign) in zip(axes, [
    ("GHG_adj_tCO2e",        "GHG adj (tCO2e)",  NEG_COL, "[−]"),
    ("Water_adj_1000m3",     "Water adj (000m³)",WAT_COL,  "[−]"),
    ("Employment_adj_FTE",   "Jobs adj (FTE)",   EMP_COL,  "[+]"),
    ("ValueAdded_adj_M$",    "VA adj (M USD)",   POS_COL,  "[+]"),
]):
    sub = sw[sw["year"] == FOCUS_YEAR]
    projects_o = sub.groupby("project_id")[col].sum().sort_values(ascending=False).index.tolist()
    x = np.arange(len(projects_o))
    w = 0.8 / n_ssps
    for i, ssp in enumerate(ssps_plot):
        ssp_vals = (sub[sub["scenario"] == ssp]
                    .set_index("project_id")[col].reindex(projects_o, fill_value=0))
        ax.bar(x + i * w, ssp_vals.values, width=w,
               color=ssp_cols.get(ssp, "#999"), alpha=0.85, label=ssp)

    ax.set_xticks(x + w * (n_ssps - 1) / 2)
    ax.set_xticklabels(projects_o, rotation=35, ha="right", fontsize=8)
    ax.set_title(f"{sign} {label} @ {FOCUS_YEAR} — SSP comparison",
                 color=color, fontweight="bold")
    ax.set_ylabel(label)
    ax.legend(fontsize=7, loc="upper right")

plt.suptitle(f"Scenario Comparison at {FOCUS_YEAR} — All Stressors",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 9. Dependency Weighting — ENCORE × WWF × SC Sensitivity

In [ ]:
# ── 9.1 dep_factor heatmap: project × stressor (SSP2-4.5, focus year) ─
dep_factors = dwt[
    (dwt["scenario"] == SCENARIO) & (dwt["year"] == FOCUS_YEAR)
].groupby("project_id").agg({
    "dep_factor_GHG_tCO2e":      "mean",
    "dep_factor_Employment_FTE":  "mean",
    "dep_factor_Water_1000m3":   "mean",
    "dep_factor_ValueAdded_M$":  "mean",
}).rename(columns={
    "dep_factor_GHG_tCO2e":     "GHG",
    "dep_factor_Employment_FTE": "Employment",
    "dep_factor_Water_1000m3":  "Water",
    "dep_factor_ValueAdded_M$": "VA",
})

fig, ax = plt.subplots(figsize=(8, 4))
cmap_dep = LinearSegmentedColormap.from_list("dep", ["#f7fbff", "#2171b5", "#08306b"])
im = ax.imshow(dep_factors.values, cmap=cmap_dep, aspect="auto", vmin=0.5, vmax=1.7)
ax.set_xticks(range(len(dep_factors.columns)))
ax.set_xticklabels(dep_factors.columns, fontsize=10)
ax.set_yticks(range(len(dep_factors.index)))
ax.set_yticklabels(dep_factors.index, fontsize=9)
for i in range(len(dep_factors.index)):
    for j in range(len(dep_factors.columns)):
        v = dep_factors.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                fontsize=9, color="white" if v > 1.2 else "black")

plt.colorbar(im, ax=ax, label="dep_factor (neutral = 1.0)", shrink=0.8)
ax.set_title(
    f"Dependency Factor — Project × Stressor ({SCENARIO}, {FOCUS_YEAR})\n"
    "dep_factor > 1.0 = ecosystem risk amplifies impact",
    fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.2 Adjusted vs dep-weighted: GHG and Water side-by-side ──────────
sub_dw = dw[(dw["scenario"] == SCENARIO) & (dw["year"] == FOCUS_YEAR)]
sub_sw = sw[(sw["scenario"] == SCENARIO) & (sw["year"] == FOCUS_YEAR)]

projs = sub_sw.sort_values("GHG_adj_tCO2e", ascending=False)["project_id"].tolist()

fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, 5))

for ax, (adj_col, dep_col, label, color, sign) in zip(axes, [
    ("GHG_adj_tCO2e",     "GHG_dep_tCO2e",    "GHG (tCO2e)",  NEG_COL, "[−]"),
    ("Water_adj_1000m3",  "Water_dep_1000m3", "Water (000m³)", WAT_COL, "[−]"),
]):
    x = np.arange(len(projs))
    adj_vals = sub_sw.set_index("project_id")[adj_col].reindex(projs, fill_value=0)
    dep_vals = sub_dw.set_index("project_id").reindex(projs, fill_value=0)
    dep_col_actual = dep_col if dep_col in dep_vals.columns else adj_col
    dep_v = dep_vals[dep_col_actual] if dep_col_actual in dep_vals.columns else adj_vals

    ax.bar(x - 0.2, adj_vals.values, width=0.38, color=color, alpha=0.6,
           label="Scenario-adjusted")
    ax.bar(x + 0.2, dep_v.values,   width=0.38, color=color, alpha=1.0,
           label="Dep-weighted")

    # Amplification annotation
    for i, (a, d) in enumerate(zip(adj_vals.values, dep_v.values)):
        if a > 0:
            ratio = d / a
            ax.text(i + 0.2, d * 1.01, f"×{ratio:.2f}",
                    ha="center", va="bottom", fontsize=7, color="#333")

    ax.set_xticks(x)
    ax.set_xticklabels(projs, rotation=35, ha="right", fontsize=8)
    ax.set_title(f"{sign} {label}: scenario-adj vs dep-weighted\n({SCENARIO}, {FOCUS_YEAR})",
                 color=color, fontweight="bold")
    ax.set_ylabel(label)
    ax.legend(fontsize=8)

plt.suptitle("Dependency Amplification — Ecosystem Risk Multiplier (×) on each stressor",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.3 dep_factor by tier and supplying sector (GHG) ─────────────────
dep_tier_sec = dwt[
    (dwt["scenario"] == SCENARIO) & (dwt["year"] == FOCUS_YEAR)
].groupby(["tier_label", "supplying_sector"])["dep_factor_GHG_tCO2e"].mean().unstack(fill_value=np.nan)

tier_ord = ["tier0", "tier1", "tier2", "tier3_10"]
dep_tier_sec = dep_tier_sec.reindex([t for t in tier_ord if t in dep_tier_sec.index])

fig, ax = plt.subplots(figsize=(10, 3.5))
cmap_dep2 = LinearSegmentedColormap.from_list("dep2", ["#ffffcc", "#fd8d3c", NEG_COL])
im = ax.imshow(dep_tier_sec.values, cmap=cmap_dep2, aspect="auto", vmin=0.5, vmax=1.7)
ax.set_xticks(range(len(dep_tier_sec.columns)))
ax.set_xticklabels(dep_tier_sec.columns, rotation=35, ha="right", fontsize=9)
ax.set_yticks(range(len(dep_tier_sec.index)))
ax.set_yticklabels([t.replace("_", "-") for t in dep_tier_sec.index], fontsize=9)
for i in range(len(dep_tier_sec.index)):
    for j in range(len(dep_tier_sec.columns)):
        v = dep_tier_sec.iloc[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if v > 1.3 else "black")

plt.colorbar(im, ax=ax, label="dep_factor GHG (neutral=1.0)", shrink=0.85)
ax.set_title(f"GHG Dependency Factor — Tier × Supplying Sector ({SCENARIO}, {FOCUS_YEAR})",
             fontweight="bold")
plt.tight_layout()
plt.show()

---
## 10. Integrated Summary

In [ ]:
# ── 10.1 Integrated positive vs negative spider chart per project ──────
from matplotlib.patches import FancyArrowPatch

sub_sw_ref = sw[(sw["scenario"] == SCENARIO) & (sw["year"] == FOCUS_YEAR)]
sub_dw_ref = dw[(dw["scenario"] == SCENARIO) & (dw["year"] == FOCUS_YEAR)]

summary_table = sc_sum[["project_id", "region", "asset_class", ghg_col, emp_col, wat_col, va_col]].copy()
summary_table.columns = ["project_id", "region", "asset_class",
                          "GHG_baseline", "Emp_baseline", "Wat_baseline", "VA_baseline"]

# Merge scenario-adjusted
adj_merge = sub_sw_ref[["project_id","GHG_adj_tCO2e","Employment_adj_FTE",
                          "Water_adj_1000m3","ValueAdded_adj_M$"]].copy()
summary_table = summary_table.merge(adj_merge, on="project_id", how="left")

# Merge dep-weighted
if "GHG_dep_tCO2e" in sub_dw_ref.columns:
    dep_merge = sub_dw_ref[["project_id","GHG_dep_tCO2e",
                             "Employment_dep_FTE","Water_dep_1000m3","ValueAdded_dep_M$"]].copy()
    summary_table = summary_table.merge(dep_merge, on="project_id", how="left")

# Merge positive outcomes
summary_table = summary_table.merge(
    pos_out[["project_id","avoided_CO2_tCO2e","beneficiaries","reach_ppl_yr"]],
    on="project_id", how="left"
)
summary_table["net_GHG"] = (
    summary_table.get("GHG_dep_tCO2e", summary_table["GHG_adj_tCO2e"])
    - summary_table["avoided_CO2_tCO2e"].fillna(0)
)

pd.set_option("display.float_format", "{:,.0f}".format)
summary_table.sort_values("GHG_baseline", ascending=False).reset_index(drop=True)

In [ ]:
# ── 10.2 Final four-quadrant plot: GHG risk vs employment benefit ──────
fig, ax = plt.subplots(figsize=(10, 7))

ghg_col_s = "GHG_dep_tCO2e" if "GHG_dep_tCO2e" in summary_table.columns else "GHG_adj_tCO2e"
emp_col_s = "Employment_dep_FTE" if "Employment_dep_FTE" in summary_table.columns else "Employment_adj_FTE"

for _, row in summary_table.iterrows():
    ghg = row.get(ghg_col_s, row["GHG_baseline"]) or row["GHG_baseline"]
    emp = row.get(emp_col_s, row["Emp_baseline"]) or row["Emp_baseline"]
    avoided = row.get("avoided_CO2_tCO2e", 0) or 0
    net_ghg = ghg - avoided
    invest  = row.get("VA_baseline", 1) or 1
    bubble  = max(np.sqrt(abs(invest)) * 15, 30)

    color = REGION_COLS.get(row["region"], "#999")
    ax.scatter(net_ghg / 1e3, emp, s=bubble, color=color, alpha=0.8,
               edgecolors="white", linewidths=0.8, zorder=3)
    ax.annotate(row["project_id"],
                xy=(net_ghg / 1e3, emp),
                xytext=(5, 3), textcoords="offset points",
                fontsize=7.5, color="#333")

ax.axvline(0, color="gray", linewidth=0.8, linestyle="--")
ax.axhline(summary_table["Emp_baseline"].median(), color="gray",
           linewidth=0.8, linestyle="--")

# Quadrant labels
xmin, xmax = ax.get_xlim(); ymin, ymax = ax.get_ylim()
ax.text(xmin * 0.9, ymax * 0.95, "Net negative GHG\nHigh employment",
        fontsize=8, color=POS_COL, alpha=0.6, ha="left")
ax.text(xmax * 0.6, ymax * 0.95, "Net positive GHG\nHigh employment",
        fontsize=8, color="#888", alpha=0.6, ha="left")
ax.text(xmax * 0.6, ymin * 0.9 if ymin < 0 else ymax * 0.05,
        "Net positive GHG\nLow employment",
        fontsize=8, color=NEG_COL, alpha=0.6, ha="left")

# Region legend
handles = [mpatches.Patch(color=c, label=r) for r, c in REGION_COLS.items()
           if r in summary_table["region"].values]
ax.legend(handles=handles, title="Region", fontsize=8, loc="lower right")

ax.set_xlabel(f"Net GHG (ktCO2e) — dep-weighted @ {SCENARIO} {FOCUS_YEAR}\n"
              "[after subtracting avoided CO₂]", fontsize=9)
ax.set_ylabel(f"Employment (FTE) — dep-weighted @ {SCENARIO} {FOCUS_YEAR}", fontsize=9)
ax.set_title(
    "Four-Quadrant Portfolio View\n"
    "[−] Net GHG Risk  vs  [+] Employment Benefit\n"
    "Bubble size ∝ Value Added (M USD)",
    fontweight="bold"
)
plt.tight_layout()
plt.show()

In [ ]:
# ── 10.3 Concise text summary ─────────────────────────────────────────
print("=" * 70)
print(f"  S4 PORTFOLIO — INTEGRATED IMPACT SUMMARY")
print(f"  Scenario: {SCENARIO}  |  Year: {FOCUS_YEAR}")
print("=" * 70)

print("\n[−] NEGATIVE IMPACTS (supply-chain, tiers 0–10)")
print(f"  GHG emissions   : {sc_sum[ghg_col].sum():>15,.0f} tCO2e  (baseline)")
ghg_adj_total = sub_sw_ref["GHG_adj_tCO2e"].sum()
print(f"                  : {ghg_adj_total:>15,.0f} tCO2e  ({SCENARIO} {FOCUS_YEAR}, scenario-adj)")
if "GHG_dep_tCO2e" in sub_dw_ref.columns:
    ghg_dep_total = sub_dw_ref["GHG_dep_tCO2e"].sum()
    print(f"                  : {ghg_dep_total:>15,.0f} tCO2e  (dep-weighted, ecosystem risk)")
print(f"  Water withdrawal: {sc_sum[wat_col].sum():>15,.0f} 000 m³  (baseline)")

print("\n[+] POSITIVE IMPACTS")
print(f"  Jobs created    : {sc_sum[emp_col].sum():>15,.0f} FTE    (supply-chain employment)")
emp_adj_total = sub_sw_ref["Employment_adj_FTE"].sum()
print(f"                  : {emp_adj_total:>15,.0f} FTE    ({SCENARIO} {FOCUS_YEAR}, renewable-jobs adj)")
print(f"  Value added     : {sc_sum[va_col].sum():>15,.2f} M USD")
print(f"  Avoided CO₂     : {pos_out['avoided_CO2_tCO2e'].sum():>15,.0f} tCO2e  (generation displacement)")
print(f"  Health benefic. : {pos_out['beneficiaries'].sum():>15,.0f} people")
print(f"  Rail reach      : {pos_out['reach_ppl_yr'].sum():>15,.0f} ppl/yr")

net = sc_sum[ghg_col].sum() - pos_out["avoided_CO2_tCO2e"].sum()
print("\n[±] NET POSITION")
print(f"  Net GHG (SC − avoided) : {net:>12,.0f} tCO2e")
dep_net = sub_dw_ref["GHG_dep_tCO2e"].sum() - pos_out["avoided_CO2_tCO2e"].sum() if "GHG_dep_tCO2e" in sub_dw_ref.columns else None
if dep_net is not None:
    print(f"  Net GHG dep-weighted   : {dep_net:>12,.0f} tCO2e  (ecosystem risk-adjusted)")
print("=" * 70)